# 00 — Setup: базовые Hive-таблицы для column lineage demo

Этот ноутбук создаёт три синтетические таблицы в `default` базе Hive, которые остальные ноутбуки используют как источники:

| Таблица           | Колонки                                              |
|-------------------|------------------------------------------------------|
| `raw_customers`   | id, name, email, country                             |
| `raw_orders`      | order_id, customer_id, amount, order_ts             |
| `raw_products`    | product_id, category, price                          |

После прогона в Marquez UI (http://localhost:3000, namespace `hadoop-cluster`) появятся три output dataset'а с initial schema version.

> ⚠️ Между каждым ноутбуком рекомендуется **рестарт Jupyter kernel** — иначе `SparkSession.builder.appName(...).getOrCreate()` может молча вернуть старую сессию и lineage events будут идти под чужим app name. Assertion ниже проверит, что переименование сработало.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_00_setup')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_00_setup', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:07:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:07:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/27 13:07:51 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:08:00 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:08:01 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:08:01 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]

app name: lineage_00_setup


In [2]:
from pyspark.sql import functions as F

customers = (
    spark.range(0, 100)
    .withColumn('name', F.concat(F.lit('user_'), F.col('id').cast('string')))
    .withColumn('email', F.concat(F.col('name'), F.lit('@example.com')))
    .withColumn('country', F.element_at(F.array(F.lit('RU'), F.lit('US'), F.lit('DE'), F.lit('FR')), (F.col('id') % 4 + 1).cast('int')))
)
customers.write.mode('overwrite').saveAsTable('default.raw_customers')
spark.table('default.raw_customers').show(5, truncate=False)

26/05/27 13:08:04 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:08:04 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:08:04 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:08:04 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
26/05/27 13:08:05 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:05 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:05 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:07 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:07 WARN InputFieldsCol

+---+-------+-------------------+-------+
|id |name   |email              |country|
+---+-------+-------------------+-------+
|50 |user_50|user_50@example.com|DE     |
|51 |user_51|user_51@example.com|FR     |
|52 |user_52|user_52@example.com|RU     |
|53 |user_53|user_53@example.com|US     |
|54 |user_54|user_54@example.com|DE     |
+---+-------+-------------------+-------+
only showing top 5 rows



In [3]:
orders = (
    spark.range(0, 500)
    .withColumnRenamed('id', 'order_id')
    .withColumn('customer_id', (F.col('order_id') % 100))
    .withColumn('amount', F.rand(seed=42) * 1000)
    .withColumn('order_ts', F.from_unixtime(F.lit(1700000000) + F.col('order_id') * 60).cast('timestamp'))
)
orders.write.mode('overwrite').saveAsTable('default.raw_orders')
spark.table('default.raw_orders').show(5, truncate=False)

26/05/27 13:08:08 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:08 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:08 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:09 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:09 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range


+--------+-----------+-----------------+-------------------+
|order_id|customer_id|amount           |order_ts           |
+--------+-----------+-----------------+-------------------+
|250     |50         |801.7532427858895|2023-11-15 02:23:20|
|251     |51         |656.5552949992319|2023-11-15 02:24:20|
|252     |52         |251.5595782593636|2023-11-15 02:25:20|
|253     |53         |207.3428376111074|2023-11-15 02:26:20|
|254     |54         |639.2921379278928|2023-11-15 02:27:20|
+--------+-----------+-----------------+-------------------+
only showing top 5 rows



26/05/27 13:08:09 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range


In [4]:
products = (
    spark.range(0, 50)
    .withColumnRenamed('id', 'product_id')
    .withColumn('category', F.element_at(F.array(F.lit('books'), F.lit('food'), F.lit('tech')), (F.col('product_id') % 3 + 1).cast('int')))
    .withColumn('price', F.rand(seed=7) * 100)
)
products.write.mode('overwrite').saveAsTable('default.raw_products')
spark.table('default.raw_products').show(5, truncate=False)

26/05/27 13:08:09 ERROR ContextFactory: Query execution is null: can't emit event for executionId 5
26/05/27 13:08:09 ERROR ContextFactory: Query execution is null: can't emit event for executionId 5
26/05/27 13:08:09 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:09 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:09 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
[Stage 4:=============================>                             (1 + 1) / 2]

+----------+--------+------------------+
|product_id|category|price             |
+----------+--------+------------------+
|25        |food    |48.36508543933039 |
|26        |tech    |73.4216416644485  |
|27        |books   |25.330679322428097|
|28        |food    |43.12578829567928 |
|29        |tech    |75.22950453235204 |
+----------+--------+------------------+
only showing top 5 rows



26/05/27 13:08:10 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:10 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range
26/05/27 13:08:10 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.plans.logical.Range


In [5]:
spark.stop()

26/05/27 13:08:10 ERROR ContextFactory: Query execution is null: can't emit event for executionId 8
26/05/27 13:08:10 ERROR ContextFactory: Query execution is null: can't emit event for executionId 8


## Проверка

```bash
curl http://localhost:5000/api/v1/namespaces/hadoop-cluster/datasets | jq '.datasets[].name'
```

Должны появиться `default.raw_customers`, `default.raw_orders`, `default.raw_products`.